In [21]:
from pypot.dynamixel import DxlIO
from pypot.dynamixel.protocol.v1 import *

from glob import glob

ports = glob('/dev/ttyACM*')
assert len(ports) == 1

port = ports[0]
print('Connecting on port: {}'.format(port))
dxl_io = DxlIO(port)

poulpe_id = 42
N_AXIS = 1

Connecting on port: /dev/ttyACM0


In [2]:
ping_packet = DxlPingPacket(poulpe_id)
dxl_io._send_packet(ping_packet)

DxlStatusPacket(id=42, error=0, parameters=())

In [3]:
import struct

def read_current_pos():
    pos_packet = DxlReadDataPacket(poulpe_id, 50, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_current_pos()

(0.0,)

In [4]:
import struct

def read_target_position():
    pos_packet = DxlReadDataPacket(poulpe_id, 60, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_target_position()

(0.0,)

In [9]:
def write_target_position(target):
    p = DxlWriteDataPacket(poulpe_id, 60, struct.pack(N_AXIS * 'f', *target))
    resp = dxl_io._send_packet(p)
    return resp

write_target_position([0.0,])

DxlStatusPacket(id=42, error=0, parameters=())

In [13]:
def read_torque_enabled():
    p = DxlReadDataPacket(poulpe_id, 40, N_AXIS)
    resp = dxl_io._send_packet(p)
    data = bytearray(resp.parameters)
    torque = struct.unpack(N_AXIS * '?', data)
    return torque

read_torque_enabled()

(True,)

In [11]:
def write_torque_enabled(torque):
    p = DxlWriteDataPacket(poulpe_id, 40, struct.pack(N_AXIS * '?', *torque))
    resp = dxl_io._send_packet(p)
    return resp

write_torque_enabled([True,])

DxlStatusPacket(id=42, error=0, parameters=())

In [34]:
import time
import numpy as np

pos = []
send_target = []
read_target = []

t0 = time.time()
while True:
    if time.time() - t0 > 10:
        break

    target = [
        #np.random.rand(), 
        45 * np.sin(2 * np.pi * 0.5 * time.time()),
    ]
    write_target_position(target)
    send_target.append(target)
    time.sleep(0.001)
    pos.append(read_current_pos())
    time.sleep(0.001)
    read_target.append(read_target_position())
    time.sleep(0.001)

In [33]:
np.array(send_target) - np.array(read_target)

array([[-8.06929741e-08],
       [-4.54233074e-07],
       [-6.06643312e-07],
       ...,
       [ 3.76962195e-08],
       [ 2.73323371e-07],
       [ 9.43091809e-07]])